# Tool使用的概述

## 1、工具的调用方式

### 1.1 方式1：直接调用


In [1]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息

    参数:
        city: 城市名称，如"北京"、"上海"

    返回:
        天气信息字符串
    """
    # 你的实现
    return city + "晴天，温度 15°C"

In [3]:
get_weather.invoke({"city":"北京"})

'北京晴天，温度 15°C'

### 1.2 方式2：基于模型进行调用

In [6]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)


In [7]:
from langchain_core.tools import tool

# 定义工具
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    # 你的实现
    return "晴天，温度 15°C"


# 绑定工具
model_with_tools = model.bind_tools([get_weather])

# AI 可以决定是否调用工具
response = model_with_tools.invoke("北京天气如何？")
# response = model_with_tools.invoke("2 + 3 = ？")

# 检查 AI 是否要调用工具
if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print("AI 直接回答：", response.content)

AI 想调用工具： [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_00_8sUKRPDzsNNJ2YUO0KsU2341', 'type': 'tool_call'}]


## 2、从Message流转看工具的调用

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)


In [6]:
from langchain.messages import HumanMessage, ToolMessage
from rich import print as rprint
from langchain_core.tools import tool

@tool
def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗~"


# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

# 声明一个消息列表
messages = [
    HumanMessage("今天北京天气如何")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加AIMessage到消息列表中
messages.append(response)

rprint(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 大模型和Agent的主要区别在于：大模型不会主动的调用工具，所以这时候我们需要主动让工具调用。
        # 返回的是ToolMessage类型消息，添加到消息列表中
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': "The user asks about Beijing weather today. I'll call the get_weather tool with city 
Beijing."
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 65,
            'prompt_tokens': 353,
            'total_tokens': 418,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 20,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
            'prompt_cache_hit_tokens': 256,
            'prompt_cache_miss_tokens': 97
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
        'id': '9601b68a-4e84-4655-b3af-cdf925758e48',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0136a-c112-7da0-bd29-a0b0d2a27fd9-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_00_rXv3BDW6jYUGHpLrrWLU4235',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 353,
        'output_tokens': 65,
        'total_tokens': 418,
        'input_token_details': {'cache_read': 256},
        'output_token_details': {'reasoning': 20}
    }
)

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_rXv3BDW6jYUGHpLrrWLU4235)
 Call ID: call_00_rXv3BDW6jYUGHpLrrWLU4235
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京天气晴朗~
=====================> messages <=====================
final_response: 
content='今天北京天气晴朗，是个适合外出活动的好天气 ☀️' additional_kwargs={'refusal': None, 'reasoning_content': ''} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 434, 'total_tokens': 449, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},

# 不使用@tool的方式定义工具

## 1、举例

In [7]:
# 1、模型的初始化
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# 2、声明一个函数（工具）
def get_weather(city : str):
    return f"{city}天气晴朗~~"

# 3、将函数绑定在模型上
model_with_tools = model.bind_tools([get_weather])

# 4、调用模型
response = model_with_tools.invoke("北京的天气怎么样")
rprint(response)

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The user is asking about the weather in Beijing. I should call the get_weather 
function with city "北京" or "Beijing". Let me use the appropriate parameter.\n\nThe user wrote in Chinese: 
"北京的天气怎么样" which means "How\'s the weather in Beijing?"\n\nLet me call the get_weather function.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 109,
            'prompt_tokens': 348,
            'total_tokens': 457,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 64,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
            'prompt_cache_hit_tokens': 256,
            'prompt_cache_miss_tokens': 92
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
        'id': '73642d8e-5cc6-4ea7-92bd-73e887be764e',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0136f-2fa6-7213-abe2-caffdc570959-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_00_4301IuipSTeA6oaJzrfG5021',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 348,
        'output_tokens': 109,
        'total_tokens': 457,
        'input_token_details': {'cache_read': 256},
        'output_token_details': {'reasoning': 64}
    }
)

## 2、工具描述的各部分详解

## 2.1 了解convert_to_openai_tool

执行`model.bind_tools([get_weather])`，底层最终会调用`convert_to_openai_tool`生成工具描述。所以我们可以直接调用后者查看解析后的工具描述。

In [8]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.2 description说明

In [9]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.3 参数说明

In [10]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    """
    查询城市的天气

    Args:
        city : 具体的城市

    Returns:
        返回城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2.4 参数类型说明

举例1：正确的

In [11]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：如下的代码运行会报错

要求：如果在docstring中声明了参数的描述，则必须在函数声明处指明参数的类型

In [19]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city:str):
#def get_weather(city):
    """
    查询城市的天气

    Args:
        city : 具体的城市
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2.5 参数默认值说明

一旦参数设置的默认值，则打印的结果中的required字段中就不再包含此参数。

举例1：

In [18]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str = "beijing"):
    """
    查询城市的天气

    Args:
        city : 具体的城市
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

In [20]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(dt:str ,city : str = "beijing"):
    """
    查询城市的天气

    Args:
        city : 具体的城市
        dt : 时间
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {
                'dt': {'description': '时间', 'type': 'string'},
                'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}
            },
            'required': ['dt'],
            'type': 'object'
        }
    }
}